In [10]:
# EXPLORE CENSUS CSV FILES
print("\n=== EXPLORING CENSUS DATA ===\n")

import pandas as pd
import os

# Check files in the directory
census_dir = "data/census/RP2022_logemt/"
if os.path.exists(census_dir):
    print(f"Files in {census_dir}:")
    for f in os.listdir(census_dir):
        size = os.path.getsize(os.path.join(census_dir, f))
        print(f"  - {f}: {size:,} bytes")
else:
    print(f"Directory not found: {census_dir}")

# Load the main data file
print("\n" + "="*60)
print("LOADING FD_LOGEMT_2022.csv")
print("="*60)

try:
    # Load with low_memory=False to handle mixed types
    df_main = pd.read_csv(
        os.path.join(census_dir, "FD_LOGEMT_2022.csv"),
        sep=';',
        encoding='utf-8',
        low_memory=False,
        nrows=10  # First, just see the structure
    )
    
    print(f"Shape: {df_main.shape}")
    print(f"\nColumns: {list(df_main.columns)}")
    print(f"\nFirst 5 rows:")
    print(df_main.head().to_string())
    
except Exception as e:
    print(f"Error: {e}")

# Load the variable description file
print("\n" + "="*60)
print("LOADING varmod_logemt_2022.csv")
print("="*60)

try:
    df_varmod = pd.read_csv(
        os.path.join(census_dir, "varmod_logemt_2022.csv"),
        sep=';',
        encoding='utf-8',
        low_memory=False
    )
    
    print(f"Shape: {df_varmod.shape}")
    print(f"\nColumns: {list(df_varmod.columns)}")
    print(f"\nFirst 20 rows (variable descriptions):")
    print(df_varmod.head(20).to_string())
    
except Exception as e:
    print(f"Error: {e}")


=== EXPLORING CENSUS DATA ===

Files in data/census/RP2022_logemt/:
  - varmod_logemt_2022.csv: 4,752,083 bytes
  - FD_LOGEMT_2022.csv: 4,951,583,998 bytes

LOADING FD_LOGEMT_2022.csv
Shape: (10, 69)

Columns: ['COMMUNE', 'ARM', 'IRIS', 'ACHL', 'AEMM', 'AEMMR', 'AGEMEN8', 'ANEM', 'ANEMR', 'ASCEN', 'BAIN', 'BATI', 'CATIRIS', 'CATL', 'CHAU', 'CHFL', 'CHOS', 'CLIM', 'CMBL', 'CUIS', 'DEROU', 'DIPLM', 'EAU', 'EGOUL', 'ELEC', 'EMPLM', 'GARL', 'HLML', 'ILETUDM', 'ILTM', 'IMMIM', 'INAIM', 'INEEM', 'INP11M', 'INP15M', 'INP17M', 'INP19M', 'INP24M', 'INP3M', 'INP60M', 'INP65M', 'INP5M', 'INP75M', 'INPAM', 'INPER', 'INPER1', 'INPER2', 'INPOM', 'INPSM', 'IPONDL', 'IRANM', 'METRODOM', 'NBPI', 'RECHM', 'REGION', 'SANI', 'SANIDOM', 'SEXEM', 'STAT_CONJM', 'STOCD', 'SURF', 'TACTM', 'TPM', 'TRANSM', 'TRIRIS', 'TYPC', 'TYPL', 'VOIT', 'WC']

First 5 rows:
   COMMUNE    ARM       IRIS  ACHL  AEMM  AEMMR  AGEMEN8  ANEM  ANEMR  ASCEN BAIN BATI CATIRIS  CATL CHAU  CHFL CHOS CLIM  CMBL CUIS DEROU  DIPLM EAU EG

In [11]:
# DECODE VARIABLE MEANINGS
print("\n=== DECODING VARIABLE MEANINGS ===\n")

import pandas as pd

# Load the variable dictionary
df_varmod = pd.read_csv(
    "data/census/RP2022_logemt/varmod_logemt_2022.csv",
    sep=';',
    encoding='utf-8'
)

# Show all unique variables
print("Variables in census data:")
for var in df_varmod['COD_VAR'].unique():
    lib = df_varmod[df_varmod['COD_VAR'] == var]['LIB_VAR'].iloc[0]
    print(f"  {var}: {lib}")

# Get detailed mapping for key variables
key_vars = ['COMMUNE', 'ACHL', 'CHAU', 'CHFL', 'TYPL', 'SURF', 'NBPI', 'CLIM']

print("\n" + "="*60)
print("KEY VARIABLE MEANINGS")
print("="*60)

for var in key_vars:
    if var in df_varmod['COD_VAR'].values:
        var_desc = df_varmod[df_varmod['COD_VAR'] == var]['LIB_VAR'].iloc[0]
        print(f"\n{var}: {var_desc}")
        
        # Get possible values
        values = df_varmod[df_varmod['COD_VAR'] == var][['COD_MOD', 'LIB_MOD']].drop_duplicates()
        if len(values) > 1:
            print("  Values:")
            for _, row in values.head(10).iterrows():
                print(f"    {row['COD_MOD']}: {row['LIB_MOD']}")


=== DECODING VARIABLE MEANINGS ===

Variables in census data:
  COMMUNE: Département et commune du lieu de résidence
  ARM: Arrondissement municipal de résidence (Paris,Lyon et Marseille)
  IRIS: Code IRIS du lieu de résidence
  ACHL: Période d'achèvement de la construction de la maison ou de l'immeuble
  AEMM: Année d'emménagement dans le logement (détaillée)
  AEMMR: Année d'emménagement dans le logement (regroupée)
  AGEMEN8: Âge regroupé de la personne de référence du ménage en 8 classes d'âge
  ANEM: Ancienneté d'emménagement dans le logement (détaillée)
  ANEMR: Ancienneté d'emménagement dans le logement (regroupée)
  ASCEN: Desserte par un ascenseur
  BAIN: Baignoire ou douche (DOM)
  BATI: Aspect du bâti (DOM)
  CATIRIS: Catégorie de l'IRIS
  CATL: Catégorie de logement
  CHAU: Moyen de chauffage du logement (DOM)
  CHFL: Chauffage central du logement (France métropolitaine)
  CHOS: Chauffe-eau solaire (DOM)
  CLIM: Existence d'au moins une pièce climatisée (DOM)
  CMBL: Combu

In [13]:
# AGGREGATE CENSUS DATA BY INSEE CODE (COMMUNE LEVEL)
print("\n=== AGGREGATING CENSUS DATA BY INSEE CODE ===\n")

import pandas as pd
import numpy as np

def aggregate_census_by_insee(file_path, sample_size=None):
    """
    Aggregate census housing data to commune (INSEE code) level.
    Returns a DataFrame with one row per INSEE code.
    """
    
    # Load data (with sample for speed)
    if sample_size:
        df = pd.read_csv(file_path, sep=';', encoding='utf-8', nrows=sample_size)
    else:
        df = pd.read_csv(file_path, sep=';', encoding='utf-8', low_memory=False)
    
    print(f"Loaded {len(df):,} housing records")
    
    # Convert string codes to numeric where needed
    if 'SURF' in df.columns:
        df['SURF_num'] = pd.to_numeric(df['SURF'], errors='coerce')
    if 'NBPI' in df.columns:
        df['NBPI_num'] = pd.to_numeric(df['NBPI'], errors='coerce')
    if 'VOIT' in df.columns:
        df['VOIT_num'] = pd.to_numeric(df['VOIT'], errors='coerce')
    
    # Filter to principal residences only (exclude second homes, vacant)
    if 'CATL' in df.columns:
        df_principal = df[df['CATL'] == 1].copy()
        print(f"Principal residences: {len(df_principal):,}")
    else:
        df_principal = df.copy()
    
    # Group by INSEE code (COMMUNE column)
    # The COMMUNE column already contains the 5-digit INSEE code
    commune_stats = df_principal.groupby('COMMUNE').agg({
        # Housing stock
        'TYPL': lambda x: (x == 1).mean(),  # % houses
        
        # Age of housing (pre-1970 = renovation potential)
        'ACHL': lambda x: (x.isin(['A11', 'A12', 'B11'])).mean(),  # % pre-1970
        
        # Heating types
        'CHFL': lambda x: (x == 3).mean() if 'CHFL' in df.columns else np.nan,  # % electric heating
        'CLIM': lambda x: (x == 1).mean() if 'CLIM' in df.columns else np.nan,  # % with AC
        
        # Size indicators
        'SURF_num': lambda x: (x >= 4).mean(),  # % >80m²
        'NBPI_num': lambda x: (x >= 4).mean(),  # % 4+ rooms
        
        # Wealth proxies
        'VOIT_num': lambda x: (x >= 2).mean(),  # % 2+ cars
        'HLML': lambda x: (x == 1).mean() if 'HLML' in df.columns else np.nan,  # % social housing
        
        # Count of housing units in this commune
        'IPONDL': 'sum' if 'IPONDL' in df.columns else 'count'
        
    }).reset_index()
    
    # Rename columns for clarity
    commune_stats.columns = [
        'insee_code',           # 5-digit INSEE commune code
        'pct_houses',           # % of housing that are houses
        'pct_pre_1970',         # % built before 1970
        'pct_electric_heating', # % with electric heating
        'pct_ac',               # % with air conditioning
        'pct_large_home_80m2',  # % >80m²
        'pct_large_home_4rooms',# % with 4+ rooms
        'pct_2plus_cars',       # % households with 2+ cars
        'pct_social_housing',   # % social housing (HLM)
        'housing_count'         # Total housing units
    ]
    
    # Add region name (useful for fallback)
    region_codes = {
        '11': 'Île-de-France',
        '24': 'Centre-Val de Loire',
        '27': 'Bourgogne-Franche-Comté',
        '28': 'Normandie',
        '32': 'Hauts-de-France',
        '44': 'Grand Est',
        '52': 'Pays de la Loire',
        '53': 'Bretagne',
        '75': 'Nouvelle-Aquitaine',
        '76': 'Occitanie',
        '84': 'Auvergne-Rhône-Alpes',
        '93': 'Provence-Alpes-Côte d\'Azur',
        '94': 'Corse'
    }
    
    # Extract department from INSEE code (first 2 digits)
    commune_stats['dept_code'] = commune_stats['insee_code'].astype(str).str[:2]
    commune_stats['region_name'] = commune_stats['dept_code'].map(region_codes).fillna('Unknown')
    
    print(f"\n✅ Aggregated to {len(commune_stats):,} communes (INSEE codes)")
    print(f"Columns: {list(commune_stats.columns)}")
    print(f"\nSample data (first 5 communes):")
    print(commune_stats.head().to_string())
    
    return commune_stats

# Run aggregation
file_path = "data/census/RP2022_logemt/FD_LOGEMT_2022.csv"
census_by_insee = aggregate_census_by_insee(file_path, sample_size=500000)

# Save for later use
census_by_insee.to_csv('census_commune_features.csv', index=False)
print("\n✅ Saved to census_commune_features.csv")


=== AGGREGATING CENSUS DATA BY INSEE CODE ===

Loaded 500,000 housing records
Principal residences: 433,539

✅ Aggregated to 1,143 communes (INSEE codes)
Columns: ['insee_code', 'pct_houses', 'pct_pre_1970', 'pct_electric_heating', 'pct_ac', 'pct_large_home_80m2', 'pct_large_home_4rooms', 'pct_2plus_cars', 'pct_social_housing', 'housing_count', 'dept_code', 'region_name']

Sample data (first 5 communes):
   insee_code  pct_houses  pct_pre_1970  pct_electric_heating  pct_ac  pct_large_home_80m2  pct_large_home_4rooms  pct_2plus_cars  pct_social_housing  housing_count dept_code region_name
0        1001    0.974576      0.322034                   0.0     0.0             0.966102               0.889831        0.649718                 0.0     354.000000        10     Unknown
1        1002    0.983471      0.661157                   0.0     0.0             0.966942               0.859504        0.520661                 0.0     121.000000        10     Unknown
2        1004    0.415965     

In [14]:
# CHECK WHAT COLUMNS ARE ACTUALLY AVAILABLE
print("\n=== CHECKING CENSUS COLUMNS ===\n")

# Load just the first row to see all columns
df_sample = pd.read_csv(file_path, sep=';', encoding='utf-8', nrows=1)
print(f"Columns in census data: {list(df_sample.columns)}")

# Check for heating and AC columns
heating_cols = ['CHFL', 'CHAU', 'CMBL', 'CLIM']
for col in heating_cols:
    if col in df_sample.columns:
        print(f"  ✅ {col} found")
    else:
        print(f"  ❌ {col} not found")

# Check region column
if 'REGION' in df_sample.columns:
    print(f"  ✅ REGION found")
else:
    print(f"  ❌ REGION not found")


=== CHECKING CENSUS COLUMNS ===

Columns in census data: ['COMMUNE', 'ARM', 'IRIS', 'ACHL', 'AEMM', 'AEMMR', 'AGEMEN8', 'ANEM', 'ANEMR', 'ASCEN', 'BAIN', 'BATI', 'CATIRIS', 'CATL', 'CHAU', 'CHFL', 'CHOS', 'CLIM', 'CMBL', 'CUIS', 'DEROU', 'DIPLM', 'EAU', 'EGOUL', 'ELEC', 'EMPLM', 'GARL', 'HLML', 'ILETUDM', 'ILTM', 'IMMIM', 'INAIM', 'INEEM', 'INP11M', 'INP15M', 'INP17M', 'INP19M', 'INP24M', 'INP3M', 'INP60M', 'INP65M', 'INP5M', 'INP75M', 'INPAM', 'INPER', 'INPER1', 'INPER2', 'INPOM', 'INPSM', 'IPONDL', 'IRANM', 'METRODOM', 'NBPI', 'RECHM', 'REGION', 'SANI', 'SANIDOM', 'SEXEM', 'STAT_CONJM', 'STOCD', 'SURF', 'TACTM', 'TPM', 'TRANSM', 'TRIRIS', 'TYPC', 'TYPL', 'VOIT', 'WC']
  ✅ CHFL found
  ✅ CHAU found
  ✅ CMBL found
  ✅ CLIM found
  ✅ REGION found


In [15]:
# CHUNKED AGGREGATION — MEMORY-SAFE
print("\n=== CHUNKED AGGREGATION BY INSEE CODE ===\n")

import pandas as pd
import numpy as np
import os

file_path = "data/census/RP2022_logemt/FD_LOGEMT_2022.csv"
chunk_size = 100000  # 100k rows per chunk (adjust based on memory)

# Initialize accumulators
accumulated_counts = {}
accumulated_sums = {}

def update_stats(chunk):
    """Process one chunk and update accumulators"""
    
    # Keep only needed columns to save memory
    needed = ['COMMUNE', 'CATL', 'TYPL', 'ACHL', 'CHFL', 'CLIM', 'SURF', 'NBPI', 'VOIT', 'HLML', 'IPONDL']
    chunk = chunk[[c for c in needed if c in chunk.columns]]
    
    # Filter to principal residences
    if 'CATL' in chunk.columns:
        chunk = chunk[chunk['CATL'] == 1]
    
    if len(chunk) == 0:
        return
    
    # Convert to numeric where needed
    if 'SURF' in chunk.columns:
        chunk['SURF_num'] = pd.to_numeric(chunk['SURF'], errors='coerce')
    if 'NBPI' in chunk.columns:
        chunk['NBPI_num'] = pd.to_numeric(chunk['NBPI'], errors='coerce')
    if 'VOIT' in chunk.columns:
        chunk['VOIT_num'] = pd.to_numeric(chunk['VOIT'], errors='coerce')
    
    # Group by commune for this chunk
    grouped = chunk.groupby('COMMUNE')
    
    for commune, group in grouped:
        if commune not in accumulated_counts:
            accumulated_counts[commune] = {
                'total': 0,
                'houses': 0,
                'pre_1970': 0,
                'electric_heating': 0,
                'ac': 0,
                'large_80m2': 0,
                'large_4rooms': 0,
                'two_cars': 0,
                'social_housing': 0,
                'weight_sum': 0
            }
        
        stats = accumulated_counts[commune]
        weight = group['IPONDL'].sum() if 'IPONDL' in group.columns else len(group)
        
        stats['total'] += weight
        stats['weight_sum'] += weight
        stats['houses'] += (group['TYPL'] == 1).sum() * (weight / len(group)) if 'TYPL' in group.columns else 0
        stats['pre_1970'] += (group['ACHL'].isin(['A11', 'A12', 'B11'])).sum() * (weight / len(group)) if 'ACHL' in group.columns else 0
        stats['electric_heating'] += (group['CHFL'] == 3).sum() * (weight / len(group)) if 'CHFL' in group.columns else 0
        stats['ac'] += (group['CLIM'] == 1).sum() * (weight / len(group)) if 'CLIM' in group.columns else 0
        stats['large_80m2'] += (group['SURF_num'] >= 4).sum() * (weight / len(group)) if 'SURF_num' in group.columns else 0
        stats['large_4rooms'] += (group['NBPI_num'] >= 4).sum() * (weight / len(group)) if 'NBPI_num' in group.columns else 0
        stats['two_cars'] += (group['VOIT_num'] >= 2).sum() * (weight / len(group)) if 'VOIT_num' in group.columns else 0
        stats['social_housing'] += (group['HLML'] == 1).sum() * (weight / len(group)) if 'HLML' in group.columns else 0

# Process file in chunks
print(f"Processing {file_path} in chunks of {chunk_size:,} rows...")

chunk_num = 0
for chunk in pd.read_csv(file_path, sep=';', encoding='utf-8', chunksize=chunk_size, low_memory=False):
    chunk_num += 1
    update_stats(chunk)
    
    if chunk_num % 10 == 0:
        print(f"  Processed {chunk_num * chunk_size:,} rows...")
    
    # Force garbage collection
    import gc
    gc.collect()

print(f"\n✅ Processed {chunk_num} chunks")
print(f"Found {len(accumulated_counts):,} communes")

# Build final dataframe
results = []
for commune, stats in accumulated_counts.items():
    if stats['total'] > 0:
        results.append({
            'insee_code': str(commune).zfill(5),
            'pct_houses': stats['houses'] / stats['total'],
            'pct_pre_1970': stats['pre_1970'] / stats['total'],
            'pct_electric_heating': stats['electric_heating'] / stats['total'],
            'pct_ac': stats['ac'] / stats['total'],
            'pct_large_home_80m2': stats['large_80m2'] / stats['total'],
            'pct_large_home_4rooms': stats['large_4rooms'] / stats['total'],
            'pct_2plus_cars': stats['two_cars'] / stats['total'],
            'pct_social_housing': stats['social_housing'] / stats['total'],
            'housing_count': stats['total']
        })

df_results = pd.DataFrame(results)
print(f"\n✅ Final dataset: {len(df_results):,} communes")

# Save
df_results.to_csv('census_commune_features.csv', index=False)
print("✅ Saved to census_commune_features.csv")

# Show sample
print("\nSample data:")
print(df_results.head(10).to_string())


=== CHUNKED AGGREGATION BY INSEE CODE ===

Processing data/census/RP2022_logemt/FD_LOGEMT_2022.csv in chunks of 100,000 rows...
  Processed 1,000,000 rows...
  Processed 2,000,000 rows...
  Processed 3,000,000 rows...
  Processed 4,000,000 rows...
  Processed 5,000,000 rows...
  Processed 6,000,000 rows...
  Processed 7,000,000 rows...
  Processed 8,000,000 rows...
  Processed 9,000,000 rows...
  Processed 10,000,000 rows...
  Processed 11,000,000 rows...
  Processed 12,000,000 rows...
  Processed 13,000,000 rows...
  Processed 14,000,000 rows...
  Processed 15,000,000 rows...
  Processed 16,000,000 rows...
  Processed 17,000,000 rows...
  Processed 18,000,000 rows...
  Processed 19,000,000 rows...
  Processed 20,000,000 rows...
  Processed 21,000,000 rows...
  Processed 22,000,000 rows...
  Processed 23,000,000 rows...
  Processed 24,000,000 rows...
  Processed 25,000,000 rows...
  Processed 26,000,000 rows...

✅ Processed 264 chunks
Found 34,914 communes

✅ Final dataset: 34,914 com

In [16]:
# INVESTIGATE HEATING AND AC CODES
print("\n=== INVESTIGATING HEATING AND AC CODES ===\n")

import pandas as pd

file_path = "data/census/RP2022_logemt/FD_LOGEMT_2022.csv"

# Read a small sample to see actual values
df_sample = pd.read_csv(file_path, sep=';', encoding='utf-8', nrows=10000)

print("CHFL (Heating) value counts:")
print(df_sample['CHFL'].value_counts().head(10))

print("\nCLIM (Air Conditioning) value counts:")
print(df_sample['CLIM'].value_counts().head(10))

print("\nCHAU (Heating presence) value counts:")
if 'CHAU' in df_sample.columns:
    print(df_sample['CHAU'].value_counts().head(10))

print("\nCMBL (Fuel type) value counts:")
if 'CMBL' in df_sample.columns:
    print(df_sample['CMBL'].value_counts().head(10))


=== INVESTIGATING HEATING AND AC CODES ===

CHFL (Heating) value counts:
CHFL
2    3914
3    2020
4    1596
Y    1267
1    1203
Name: count, dtype: int64

CLIM (Air Conditioning) value counts:
CLIM
Z    10000
Name: count, dtype: int64

CHAU (Heating presence) value counts:
CHAU
Z    10000
Name: count, dtype: int64

CMBL (Fuel type) value counts:
CMBL
4    2597
2    2432
6    2079
Y    1267
3    1208
5     220
1     197
Name: count, dtype: int64


In [18]:
# RECALCULATE HEATING STATISTICS WITH CORRECT CODES AND MERGE
print("\n=== RECALCULATING HEATING STATISTICS ===\n")

import pandas as pd
import gc
from tqdm import tqdm

file_path = "data/census/RP2022_logemt/FD_LOGEMT_2022.csv"
chunk_size = 100000

# Initialize accumulators
heating_stats = {}

def update_heating_stats(chunk):
    """Process one chunk for heating statistics"""
    
    # Keep only needed columns
    needed = ['COMMUNE', 'CATL', 'CHFL', 'IPONDL']
    chunk = chunk[[c for c in needed if c in chunk.columns]]
    
    # Filter to principal residences
    if 'CATL' in chunk.columns:
        chunk = chunk[chunk['CATL'] == 1]
    
    if len(chunk) == 0:
        return
    
    # Group by commune
    grouped = chunk.groupby('COMMUNE')
    
    for commune, group in grouped:
        if commune not in heating_stats:
            heating_stats[commune] = {
                'total': 0,
                'electric': 0,
                'gas': 0,
                'collective': 0,
                'other': 0
            }
        
        stats = heating_stats[commune]
        weight = group['IPONDL'].sum() if 'IPONDL' in group.columns else len(group)
        
        stats['total'] += weight
        
        # CHFL codes:
        # 1 = Collective central heating
        # 2 = Individual central heating (gas)
        # 3 = Electric heating
        # 4 = Other
        if 'CHFL' in group.columns:
            stats['collective'] += (group['CHFL'] == 1).sum() * (weight / len(group))
            stats['gas'] += (group['CHFL'] == 2).sum() * (weight / len(group))
            stats['electric'] += (group['CHFL'] == 3).sum() * (weight / len(group))
            stats['other'] += (group['CHFL'] == 4).sum() * (weight / len(group))

# Process file in chunks
print("Processing heating data...")
with tqdm(desc="Processing chunks", total=264) as pbar:
    for chunk in pd.read_csv(file_path, sep=';', encoding='utf-8', chunksize=chunk_size, low_memory=False):
        update_heating_stats(chunk)
        pbar.update(1)
        if pbar.n % 10 == 0:
            gc.collect()

print(f"\n✅ Processed {pbar.n} chunks")
print(f"Found {len(heating_stats):,} communes")

# Build results dataframe
heating_results = []
for commune, stats in heating_stats.items():
    if stats['total'] > 0:
        heating_results.append({
            'insee_code': str(commune).zfill(5),
            'pct_electric_heating': stats['electric'] / stats['total'],
            'pct_gas_heating': stats['gas'] / stats['total'],
            'pct_collective_heating': stats['collective'] / stats['total'],
            'pct_other_heating': stats['other'] / stats['total'],
            'heating_count': stats['total']
        })

df_heating = pd.DataFrame(heating_results)
print(f"\n✅ Heating data: {len(df_heating):,} communes")

# Load existing census data
print("\nLoading existing census data...")
census_full = pd.read_csv('census_commune_features.csv')
print(f"Loaded {len(census_full):,} communes")

# Merge on insee_code
census_full = census_full.merge(df_heating, on='insee_code', how='left')
print(f"After merge: {len(census_full):,} communes")

# Fill NaN with 0 for heating columns
heating_cols = ['pct_electric_heating', 'pct_gas_heating', 'pct_collective_heating', 'pct_other_heating']
for col in heating_cols:
    if col in census_full.columns:
        census_full[col] = census_full[col].fillna(0)
    else:
        print(f"⚠️ Column {col} not found after merge")

# Save updated file
census_full.to_csv('census_commune_features.csv', index=False)
print("✅ Updated census_commune_features.csv with correct heating data")

# Show sample
print("\nSample of updated data:")
sample_cols = ['insee_code', 'pct_houses', 'pct_electric_heating', 'pct_gas_heating', 'pct_collective_heating']
print(census_full[sample_cols].head(10).to_string())


=== RECALCULATING HEATING STATISTICS ===

Processing heating data...


Processing chunks: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [03:55<00:00,  1.12it/s]



✅ Processed 264 chunks
Found 34,914 communes

✅ Heating data: 34,914 communes

Loading existing census data...
Loaded 34,918 communes
After merge: 34,926 communes
⚠️ Column pct_gas_heating not found after merge
⚠️ Column pct_collective_heating not found after merge
⚠️ Column pct_other_heating not found after merge
✅ Updated census_commune_features.csv with correct heating data

Sample of updated data:


KeyError: "['pct_gas_heating', 'pct_collective_heating'] not in index"

In [19]:
# CHECK WHAT COLUMNS ARE IN df_heating
print("\n=== CHECKING HEATING DATAFRAME COLUMNS ===\n")
print("df_heating columns:", df_heating.columns.tolist())
print("\nFirst 3 rows of df_heating:")
print(df_heating.head(3))

# Also check census_full columns after merge
print("\n=== CENSUS FULL COLUMNS AFTER MERGE ===\n")
print("census_full columns:", census_full.columns.tolist())


=== CHECKING HEATING DATAFRAME COLUMNS ===

df_heating columns: ['insee_code', 'pct_electric_heating', 'pct_gas_heating', 'pct_collective_heating', 'pct_other_heating', 'heating_count']

First 3 rows of df_heating:
  insee_code  pct_electric_heating  pct_gas_heating  pct_collective_heating  \
0      01001                   0.0              0.0                     0.0   
1      01002                   0.0              0.0                     0.0   
2      01004                   0.0              0.0                     0.0   

   pct_other_heating  heating_count  
0                0.0     354.000000  
1                0.0     121.000000  
2                0.0    7107.012476  

=== CENSUS FULL COLUMNS AFTER MERGE ===

census_full columns: ['insee_code', 'pct_houses', 'pct_pre_1970', 'pct_electric_heating_x', 'pct_ac', 'pct_large_home_80m2', 'pct_large_home_4rooms', 'pct_2plus_cars', 'pct_social_housing', 'housing_count', 'pct_electric_heating_y', 'pct_gas_heating_x', 'pct_collective_hea

In [20]:
# CREATE COMPLETE CENSUS FILE FROM SCRATCH
print("\n=== CREATING COMPLETE CENSUS FILE ===\n")

import pandas as pd
import gc
from tqdm import tqdm

file_path = "data/census/RP2022_logemt/FD_LOGEMT_2022.csv"
chunk_size = 100000

# Initialize accumulators
census_data = {}

def update_census_stats(chunk):
    """Process one chunk and accumulate all stats"""
    
    # Filter to principal residences
    if 'CATL' in chunk.columns:
        chunk = chunk[chunk['CATL'] == 1]
    
    if len(chunk) == 0:
        return
    
    # Convert numeric columns
    if 'SURF' in chunk.columns:
        chunk['SURF_num'] = pd.to_numeric(chunk['SURF'], errors='coerce')
    if 'NBPI' in chunk.columns:
        chunk['NBPI_num'] = pd.to_numeric(chunk['NBPI'], errors='coerce')
    if 'VOIT' in chunk.columns:
        chunk['VOIT_num'] = pd.to_numeric(chunk['VOIT'], errors='coerce')
    
    # Group by commune
    grouped = chunk.groupby('COMMUNE')
    
    for commune, group in grouped:
        if commune not in census_data:
            census_data[commune] = {
                'total_weight': 0,
                'houses': 0,
                'pre_1970': 0,
                'electric_heating': 0,
                'gas_heating': 0,
                'collective_heating': 0,
                'other_heating': 0,
                'ac': 0,
                'large_80m2': 0,
                'large_4rooms': 0,
                'two_cars': 0,
                'social_housing': 0
            }
        
        stats = census_data[commune]
        weight = group['IPONDL'].sum() if 'IPONDL' in group.columns else len(group)
        
        stats['total_weight'] += weight
        
        # Housing type
        if 'TYPL' in group.columns:
            stats['houses'] += (group['TYPL'] == 1).sum() * (weight / len(group))
        
        # Age
        if 'ACHL' in group.columns:
            stats['pre_1970'] += (group['ACHL'].isin(['A11', 'A12', 'B11'])).sum() * (weight / len(group))
        
        # Heating (CHFL codes)
        if 'CHFL' in group.columns:
            stats['collective_heating'] += (group['CHFL'] == 1).sum() * (weight / len(group))
            stats['gas_heating'] += (group['CHFL'] == 2).sum() * (weight / len(group))
            stats['electric_heating'] += (group['CHFL'] == 3).sum() * (weight / len(group))
            stats['other_heating'] += (group['CHFL'] == 4).sum() * (weight / len(group))
        
        # AC
        if 'CLIM' in group.columns:
            stats['ac'] += (group['CLIM'] == 1).sum() * (weight / len(group))
        
        # Size
        if 'SURF_num' in group.columns:
            stats['large_80m2'] += (group['SURF_num'] >= 4).sum() * (weight / len(group))
        if 'NBPI_num' in group.columns:
            stats['large_4rooms'] += (group['NBPI_num'] >= 4).sum() * (weight / len(group))
        
        # Wealth
        if 'VOIT_num' in group.columns:
            stats['two_cars'] += (group['VOIT_num'] >= 2).sum() * (weight / len(group))
        
        # Social housing
        if 'HLML' in group.columns:
            stats['social_housing'] += (group['HLML'] == 1).sum() * (weight / len(group))

# Process file in chunks
print("Processing census data...")
with tqdm(desc="Processing chunks", total=264) as pbar:
    for chunk in pd.read_csv(file_path, sep=';', encoding='utf-8', chunksize=chunk_size, low_memory=False):
        update_census_stats(chunk)
        pbar.update(1)
        if pbar.n % 10 == 0:
            gc.collect()

print(f"\n✅ Processed {pbar.n} chunks")
print(f"Found {len(census_data):,} communes")

# Build final dataframe
results = []
for commune, stats in census_data.items():
    if stats['total_weight'] > 0:
        results.append({
            'insee_code': str(commune).zfill(5),
            'pct_houses': stats['houses'] / stats['total_weight'],
            'pct_pre_1970': stats['pre_1970'] / stats['total_weight'],
            'pct_electric_heating': stats['electric_heating'] / stats['total_weight'],
            'pct_gas_heating': stats['gas_heating'] / stats['total_weight'],
            'pct_collective_heating': stats['collective_heating'] / stats['total_weight'],
            'pct_other_heating': stats['other_heating'] / stats['total_weight'],
            'pct_ac': stats['ac'] / stats['total_weight'],
            'pct_large_home_80m2': stats['large_80m2'] / stats['total_weight'],
            'pct_large_home_4rooms': stats['large_4rooms'] / stats['total_weight'],
            'pct_2plus_cars': stats['two_cars'] / stats['total_weight'],
            'pct_social_housing': stats['social_housing'] / stats['total_weight'],
            'housing_count': stats['total_weight']
        })

df_census = pd.DataFrame(results)
print(f"\n✅ Final census dataset: {len(df_census):,} communes")
print(f"Columns: {list(df_census.columns)}")

# Save
df_census.to_csv('census_commune_features.csv', index=False)
print("✅ Saved to census_commune_features.csv")

# Show sample
print("\nSample of complete census data:")
print(df_census.head(10).to_string())


=== CREATING COMPLETE CENSUS FILE ===

Processing census data...


Processing chunks:   0%|                                                                                                                                                                           | 0/264 [00:00<?, ?it/s]/tmp/ipykernel_46809/1317066064.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chunk['SURF_num'] = pd.to_numeric(chunk['SURF'], errors='coerce')
/tmp/ipykernel_46809/1317066064.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chunk['NBPI_num'] = pd.to_numeric(chunk['NBPI'], errors='


✅ Processed 264 chunks
Found 34,914 communes

✅ Final census dataset: 34,914 communes
Columns: ['insee_code', 'pct_houses', 'pct_pre_1970', 'pct_electric_heating', 'pct_gas_heating', 'pct_collective_heating', 'pct_other_heating', 'pct_ac', 'pct_large_home_80m2', 'pct_large_home_4rooms', 'pct_2plus_cars', 'pct_social_housing', 'housing_count']
✅ Saved to census_commune_features.csv

Sample of complete census data:
  insee_code  pct_houses  pct_pre_1970  pct_electric_heating  pct_gas_heating  pct_collective_heating  pct_other_heating  pct_ac  pct_large_home_80m2  pct_large_home_4rooms  pct_2plus_cars  pct_social_housing  housing_count
0      01001    0.974576      0.322034                   0.0              0.0                     0.0                0.0     0.0             0.966102               0.889831        0.649718                 0.0     354.000000
1      01002    0.983471      0.661157                   0.0              0.0                     0.0                0.0     0.0      

In [21]:
# AGGREGATE CENSUS DATA BY REGION
print("\n=== AGGREGATING CENSUS DATA BY REGION ===\n")

import pandas as pd

# Load census data
census = pd.read_csv('census_commune_features.csv')
print(f"Loaded {len(census):,} communes")

# Add region mapping from department code
region_codes = {
    '11': 'Île-de-France',
    '24': 'Centre-Val de Loire',
    '27': 'Bourgogne-Franche-Comté',
    '28': 'Normandie',
    '32': 'Hauts-de-France',
    '44': 'Grand Est',
    '52': 'Pays de la Loire',
    '53': 'Bretagne',
    '75': 'Nouvelle-Aquitaine',
    '76': 'Occitanie',
    '84': 'Auvergne-Rhône-Alpes',
    '93': 'Provence-Alpes-Côte d\'Azur',
    '94': 'Corse'
}

# Extract department code (first 2 digits of INSEE)
census['dept_code'] = census['insee_code'].astype(str).str[:2]
census['region_name'] = census['dept_code'].map(region_codes).fillna('Unknown')

# Group by region
census_by_region = census.groupby('region_name').agg({
    'pct_houses': 'mean',
    'pct_pre_1970': 'mean',
    'pct_electric_heating': 'mean',
    'pct_gas_heating': 'mean',
    'pct_collective_heating': 'mean',
    'pct_other_heating': 'mean',
    'pct_ac': 'mean',
    'pct_large_home_80m2': 'mean',
    'pct_large_home_4rooms': 'mean',
    'pct_2plus_cars': 'mean',
    'pct_social_housing': 'mean',
    'housing_count': 'sum'
}).reset_index()

print("\n✅ Census data aggregated by region:")
print(census_by_region.to_string())


=== AGGREGATING CENSUS DATA BY REGION ===

Loaded 34,914 communes

✅ Census data aggregated by region:
                   region_name  pct_houses  pct_pre_1970  pct_electric_heating  pct_gas_heating  pct_collective_heating  pct_other_heating  pct_ac  pct_large_home_80m2  pct_large_home_4rooms  pct_2plus_cars  pct_social_housing  housing_count
0         Auvergne-Rhône-Alpes    0.837924      0.366486                   0.0              0.0                     0.0                0.0     0.0             0.862476               0.735893        0.487691                 0.0   2.595838e+05
1      Bourgogne-Franche-Comté    0.956845      0.403678                   0.0              0.0                     0.0                0.0     0.0             0.924609               0.833716        0.562710                 0.0   2.611931e+05
2                     Bretagne    0.963575      0.485918                   0.0              0.0                     0.0                0.0     0.0             0.918871   